In [2]:
%pip install trafilatura spacy pandas httpx
!python -m spacy download en_core_web_trf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 73.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.0/875.0 kB 59.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 84.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 88.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 88.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 81.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 75.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 50.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 72.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.6/803.6 kB 48.1 MB/s  0:00:00
   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  5/36 [tqdm]  WARNING: The script tqdm is installed in '/usr/local/python/3.12.1/bin' which is not on PATH.
  Con

In [3]:
import trafilatura
import json
from typing import Optional, List

# -----------------------------
# Configuration
# -----------------------------
MIN_WORDS = 500
OUTPUT_FILE = "architecture_corpus.jsonl"

URLS = [
    "https://whc.unesco.org/en/list/1321/",
    "https://smarthistory.org/curated-guide/global-history-of-architecture-syllabus/",
    "https://www.britannica.com/place/Fallingwater",
    "https://www.architecturecourses.org/history/complete-architectural-styles-timeline#toc-6-modern-architecture-1900-1970s",
    "https://www.fondationlecorbusier.fr/oeuvre-livre/vers-une-architecture-le-corbusier-1923/",
    "https://www.academia.edu/96833138/The_Role_of_the_Fondation_Le_Corbusier_in_the_Conservation_of_the_Le_Corbusiers_Architectural_Work",
    "https://theses.hal.science/tel-05045378v1/document",
    "https://www.proprietesdecharme.com/2024/01/28/architecture-belle-epoque-france-style/",
    "https://www.ifmag.fr/un-temoignage-architectural-qui-raconte-une-epoque-avec-elegance/",
    "https://fr.hisour.com/fr/données/architecture-parisienne-de-la-belle-époque/",
    "https://www.villa-ephrussi.com/fr/un-peu-dhistoire",
    "https://www.academiedesbeauxarts.fr/la-villa-et-les-jardins-ephrussi-de-rothschild",
    "https://www.institutdefrance.fr/lepatrimoine/villa-grecque-kerylos/",
    "https://www.villakerylos.fr/la-villa-kerylos/"
]

# -----------------------------
# Utility functions
# -----------------------------
def is_useful(text: str, min_words: int = MIN_WORDS) -> bool:
    """
    Checks whether extracted text is considered useful.
    """
    if not text:
        return False
    return len(text.split()) >= min_words


def extract_main_text(url: str) -> Optional[str]:
    """
    Fetches and extracts the main textual content of a webpage using trafilatura.
    """
    downloaded = trafilatura.fetch_url(url)
    if not downloaded:
        return None

    text = trafilatura.extract(
        downloaded,
        include_comments=False,
        include_tables=False,
        no_fallback=True
    )
    return text


def save_to_jsonl(records: List[dict], output_path: str):
    """
    Saves records to a JSONL file.
    """
    with open(output_path, "w", encoding="utf-8") as f:
        for record in records:
            json.dump(record, f, ensure_ascii=False)
            f.write("\n")

# -----------------------------
# Main pipeline
# -----------------------------
def build_corpus(urls: List[str]) -> None:
    corpus = []

    for url in urls:
        print(f"Processing: {url}")
        text = extract_main_text(url)

        if text and is_useful(text):
            corpus.append({
                "url": url,
                "word_count": len(text.split()),
                "text": text
            })
            print("  -> Saved")
        else:
            print("  -> Skipped (not useful or extraction failed)")

    save_to_jsonl(corpus, OUTPUT_FILE)
    print(f"\nSaved {len(corpus)} documents to {OUTPUT_FILE}")


if __name__ == "__main__":
    build_corpus(URLS)


Processing: https://whc.unesco.org/en/list/1321/
  -> Skipped (not useful or extraction failed)
Processing: https://smarthistory.org/curated-guide/global-history-of-architecture-syllabus/
  -> Saved
Processing: https://www.britannica.com/place/Fallingwater
  -> Skipped (not useful or extraction failed)
Processing: https://www.architecturecourses.org/history/complete-architectural-styles-timeline#toc-6-modern-architecture-1900-1970s
  -> Skipped (not useful or extraction failed)
Processing: https://www.fondationlecorbusier.fr/oeuvre-livre/vers-une-architecture-le-corbusier-1923/
  -> Saved
Processing: https://www.academia.edu/96833138/The_Role_of_the_Fondation_Le_Corbusier_in_the_Conservation_of_the_Le_Corbusiers_Architectural_Work
  -> Saved
Processing: https://theses.hal.science/tel-05045378v1/document
  -> Skipped (not useful or extraction failed)
Processing: https://www.proprietesdecharme.com/2024/01/28/architecture-belle-epoque-france-style/
  -> Saved
Processing: https://www.ifmag

## Phase 2: Information Extraction (90 Mins)
2.1 Named Entity Recognition (NER)

In [5]:
%pip install spacy
!python -m spacy download en_core_web_trf
!python -m spacy download fr_core_news_md

Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 46.7 MB/s  0:00:06:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 81.2 MB/s  0:00:00m0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')


In [2]:
import json
import spacy
from collections import defaultdict
from typing import Dict, Set

# -----------------------------
# Load spaCy models
# -----------------------------
NLP_MODELS = {
    "en": spacy.load("en_core_web_trf"),
    "fr": spacy.load("fr_core_news_md")
}

TARGET_LABELS = {
    "PERSON",
    "ORG",
    "GPE",
    "DATE",
    "WORK_OF_ART",
    "EVENT"
}

STOP_ENTITIES = {
    "PERSON": {"architect", "author"},
    "ORG": {"museum", "foundation", "company", "organization"},
    "GPE": {"country", "city", "region"},
    "DATE": {"today", "now"},
    "WORK_OF_ART": {"book", "building"},
    "EVENT": {"exhibition", "conference"}
}

# -----------------------------
# Helpers
# -----------------------------
def detect_language(text: str) -> str:
    if any(word in text.lower() for word in [" le ", " la ", " les ", " française "]):
        return "fr"
    return "en"


def is_valid_entity(ent) -> bool:
    text = ent.text.strip()

    if len(text) < 3:
        return False

    if ent.label_ not in TARGET_LABELS:
        return False

    if text.lower() in STOP_ENTITIES.get(ent.label_, set()):
        return False

    if ent.label_ in {"PERSON", "ORG", "GPE", "WORK_OF_ART", "EVENT"}:
        if not any(tok.is_title for tok in ent):
            return False

    return True


# -----------------------------
# Main extraction
# -----------------------------
def extract_nodes(jsonl_path: str) -> Dict[str, Set[str]]:
    nodes = defaultdict(set)

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            text = record["text"]

            lang = detect_language(text)
            nlp = NLP_MODELS[lang]
            doc = nlp(text)

            for ent in doc.ents:
                if is_valid_entity(ent):
                    nodes[ent.label_].add(ent.text.strip())

    return nodes


if __name__ == "__main__":
    nodes = extract_nodes("architecture_corpus.jsonl")

    for label, values in nodes.items():
        print(f"\n{label} ({len(values)} nodes)")
        for v in sorted(values)[:10]:
            print("  -", v)


ORG (112 nodes)
  - Academia Medicine
  - Académie des Beaux-Arts
  - Académie des Inscriptions et Belles-Lettres
  - Académie des beaux-arts
  - Académie des inscriptions et belles-lettres
  - Although the relationship between environmental
  - Argument
  - Art Nouveau
  - Art and architecture of Saint Catherine’s Monastery at Mount Sinai
  - Art and architecture of the Vijayanagara empire


2.2 Introduction to Relations

In [ ]:
import json
import spacy
from typing import List, Dict

# -----------------------------
# Load models
# -----------------------------
NLP = {
    "en": spacy.load("en_core_web_trf"),
    "fr": spacy.load("fr_core_news_md")
}

TARGET_ENTS = {"PERSON", "ORG", "GPE", "DATE"}

# -----------------------------
# Helpers
# -----------------------------
def detect_language(text: str) -> str:
    return "fr" if any(w in text.lower() for w in [" le ", " la ", " les "]) else "en"


def get_entity_by_token(token, doc):
    for ent in doc.ents:
        if token.i >= ent.start and token.i < ent.end:
            if ent.label_ in TARGET_ENTS:
                return ent
    return None


# -----------------------------
# Edge extraction
# -----------------------------
def extract_edges(jsonl_path: str) -> List[Dict]:
    edges = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            text = record["text"]

            nlp = NLP[detect_language(text)]
            doc = nlp(text)

            for sent in doc.sents:
                for token in sent:
                    # Identify a verb
                    if token.pos_ == "VERB":
                        subj = None
                        obj = None

                        for child in token.children:
                            if child.dep_ in {"nsubj", "nsubjpass"}:
                                subj = get_entity_by_token(child, doc)
                            elif child.dep_ in {"dobj", "pobj", "obj"}:
                                obj = get_entity_by_token(child, doc)

                        if subj and obj:
                            edges.append({
                                "source": subj.text,
                                "relation": token.lemma_,
                                "target": obj.text,
                                "sentence": sent.text.strip()
                            })

    return edges


if __name__ == "__main__":
    edges = extract_edges("architecture_corpus.jsonl")

    print(f"\n{'='*70}")
    print(f"Total relations extracted: {len(edges)}")
    print(f"{'='*70}\n")

    print("Top 10 Relations:\n")
    for i, e in enumerate(edges[:10], 1):
        print(f"{i}. ({e['source']}) -> [{e['relation']}] -> ({e['target']})")


Total relations extracted: 1

Top 10 Relations:

1. (Canada and the United States) -> [coronary] -> (Canada and the United States)
   Context: after percutaneous coronary interventions performed in Canada and the United States: A brief report ...

